**KLE INTERACTIVE EXPLORER**

Explorez KLE avec différentes fonctions de covariance.


In [3]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

import sys
from pathlib import Path

def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "boed" / "__init__.py").is_file():
            return p
        if (p / "pyBOED" / "boed" / "__init__.py").is_file():
            return p / "pyBOED"
    return cwd

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for name in list(sys.modules):
    if name == "boed" or name.startswith("boed."):
        del sys.modules[name]

repo_root = PROJECT_ROOT
print(f"Using PROJECT_ROOT={PROJECT_ROOT}")

from boed.reduction.methods import KLE
from boed.reduction.methods import gaussian_covariance, \
matern_32_covariance, matern_52_covariance, delta_covariance, spherical_covariance

Using PROJECT_ROOT=/home/mdoumbou/Documents/Biblio_thèse/pyBOED


In [4]:
# ============================================================================
# WIDGETS
# ============================================================================

# Paramètres de discrétisation
n_points_slider = widgets.IntSlider(
    value=101,
    min=50,
    max=10000,
    step=50,
    description='# Points:',
    continuous_update=False
)

# Paramètres de covariance
cov_type_dropdown = widgets.Dropdown(
    options=[
        ('Exponential', 'exp'),
        ('Gaussian (RBF)', 'gaus'),
        ('Matern v=3/2', 'matern32'),
        ('Matern v=5/2', 'matern52'),
        ('delta cov', 'delta'),
        ('spherical', 'sphe')
    ],
    value='exp',
    description='Covariance:'
)

# Paramètres de covariance
quadrature_type_dropdown = widgets.Dropdown(
    options=[
        'trapezoid',
        'simpson'
    ],
    value='trapezoid',
    description='Quadrature:'
)

length_scale_slider = widgets.FloatLogSlider(
    value=0.1,
    base=10,
    min=-2,  # 10^-2 = 0.01
    max=0,   # 10^0 = 1
    step=0.1,
    description='Length scale:',
    continuous_update=False
)

variance_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=5.0,
    step=0.1,
    description='Variance:',
    continuous_update=False
)

# Paramètres de visualisation
n_modes_slider = widgets.IntSlider(
    value=6,
    min=1,
    max=20,
    step=1,
    description='# Modes show:',
    continuous_update=False
)

n_samples_kle_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description='# Samples:',
    continuous_update=False
)

rank_slider = widgets.IntSlider(
    value=10,
    min=1,
    max=50,
    step=1,
    description='Rank (r):',
    continuous_update=False
)

compute_button_kle = widgets.Button(
    description='Compute KLE',
    button_style='success',
    icon='play'
)

output_kle = widgets.Output()

In [5]:
# ============================================================================
# FONCTIONS
# ============================================================================

def get_covariance_function(cov_type, length_scale, variance):
    """Retourne la fonction de covariance sélectionnée."""
    
    if cov_type == 'exp':
        def cov_func(z, zp):
            return exponential_covariance(z, zp, length_scale, variance)
    
    elif cov_type == 'gaus':
        def cov_func(z, zp):
            return gaussian_covariance(z, zp, length_scale, variance)
    
    elif cov_type == 'matern32':
        def cov_func(z, zp):
            return matern_32_covariance(z, zp, length_scale, variance)
    
    elif cov_type == 'matern52':
        def cov_func(z, zp):
            return matern_52_covariance(z, zp, length_scale, variance)
    elif cov_type == 'delta':
        def cov_func(z, zp):
            return delta_covariance(z, zp, variance)
    elif cov_type == 'sphe':
        def cov_func(z, zp):
            return spherical_covariance(z, zp, length_scale, variance)
    
    return cov_func


def compute_kle_interactive(button=None):
    """Calcule KLE et visualise."""
    
    with output_kle:
        clear_output(wait=True)
        
        print("⏳ Construction KLE...")
        
        # Créer KLE
        kle = KLE(
            domain=[0, 1],
            n_points=n_points_slider.value,
            quadrature=quadrature_type_dropdown.value
        )
        
        # Fonction de covariance
        cov_func = get_covariance_function(
            cov_type_dropdown.value,
            length_scale_slider.value,
            variance_slider.value
        )
        
        # Fit
        kle.fit(cov_func)
        
        print(f"✓ KLE calculée: {len(kle.eigenvalues)} modes")
        
        # ====================================================================
        # VISUALISATION
        # ====================================================================
        
        fig = make_subplots(
            rows=3, cols=2,
            subplot_titles=(
                'Eigenvalue Decay',
                f'First {n_modes_slider.value} Eigenfunctions',
                'Covariance Function c(z,z\')',
                'Random Field Realizations',
                'Variance Convergence',
                'Point-wise Variance'
            ),
            specs=[
                [{'type': 'scatter'}, {'type': 'scatter'}],
                [{'type': 'heatmap'}, {'type': 'scatter'}],
                [{'type': 'scatter'}, {'type': 'scatter'}]
            ],
            vertical_spacing=0.12,
            horizontal_spacing=0.15
        )
        
        # SUBPLOT 1 : Eigenvalues
        fig.add_trace(
            go.Scatter(
                x=list(range(1, len(kle.eigenvalues) + 1)),
                y=kle.eigenvalues,
                mode='markers+lines',
                marker=dict(size=6, color='blue'),
                name='λᵢ'
            ),
            row=1, col=1
        )

        fig.add_vline(
            x=rank_slider.value, 
            line_width=2, 
            line_dash="dash", 
            line_color="red",
            annotation_text=f"Rank={rank_slider.value}",
            annotation_position="top right",
            row=1, col=1
        )
        
        fig.update_yaxes(type='log', title_text='Eigenvalue', row=1, col=1)
        fig.update_xaxes(title_text='Mode i', row=1, col=1)
        
        # SUBPLOT 2 : Eigenfunctions
        import plotly.express as px
        colors = px.colors.qualitative.Set2
        
        for i in range(min(n_modes_slider.value, kle.eigenfunctions.shape[1])):
            fig.add_trace(
                go.Scatter(
                    x=kle.z,
                    y=kle.eigenfunctions[:, i],
                    mode='lines',
                    name=f'φ_{i+1}',
                    line=dict(color=colors[i % len(colors)], width=2)
                ),
                row=1, col=2
            )
        
        fig.update_xaxes(title_text='z', row=1, col=2)
        fig.update_yaxes(title_text='φᵢ(z)', row=1, col=2)
        
        # SUBPLOT 3 : Covariance matrix
        fig.add_trace(
            go.Heatmap(
                z=kle.C_matrix,
                x=kle.z,
                y=kle.z,
                colorscale='Blues',
                showscale=True
            ),
            row=2, col=1
        )
        
        fig.update_xaxes(title_text='z\'', row=2, col=1)
        fig.update_yaxes(title_text='z', row=2, col=1)
        
        # SUBPLOT 4 : Random realizations
        samples, _ = kle.sample(
            n_samples=n_samples_kle_slider.value,
            rank=rank_slider.value,
            random_state=42
        )
        
        for i in range(n_samples_kle_slider.value):
            fig.add_trace(
                go.Scatter(
                    x=kle.z,
                    y=samples[i],
                    mode='lines',
                    name=f'Sample {i+1}',
                    opacity=0.7
                ),
                row=2, col=2
            )
        
        fig.update_xaxes(title_text='z', row=2, col=2)
        fig.update_yaxes(title_text='a(z)', row=2, col=2)
        
        # SUBPLOT 5 : Variance convergence
        total_var = kle.eigenvalues.sum()
        cumsum = np.cumsum(kle.eigenvalues) / total_var * 100
        
        fig.add_trace(
            go.Scatter(
                x=list(range(1, len(cumsum) + 1)),
                y=cumsum,
                mode='lines',
                fill='tozeroy',
                line=dict(color='green', width=3),
                name='Cumulative'
            ),
            row=3, col=1
        )

        # Dans la partie SUBPLOT 5 : Variance convergence
        current_variance = cumsum[rank_slider.value - 1]

        fig.add_vline(
            x=rank_slider.value, 
            line_width=2, 
            line_dash="dash", 
            line_color="red",
            row=3, col=1
        )

        fig.add_annotation(
            x=rank_slider.value, 
            y=current_variance,
            text=f"{current_variance:.1f}%",
            showarrow=True,
            arrowhead=1,
            row=3, col=1
        )
                
        fig.update_xaxes(title_text='# Modes', row=3, col=1)
        fig.update_yaxes(title_text='Variance (%)', row=3, col=1)
        
        # SUBPLOT 6 : Point-wise variance
        pointwise_var = np.sum([kle.eigenvalues[i] * kle.eigenfunctions[:, i]**2 
                                for i in range(len(kle.eigenvalues))], axis=0)
        
        fig.add_trace(
            go.Scatter(
                x=kle.z,
                y=pointwise_var,
                mode='lines',
                fill='tozeroy',
                line=dict(color='purple', width=2),
                name='From KLE'
            ),
            row=3, col=2
        )
        
        # Theoretical
        theoretical_var = np.diag(kle.C_matrix)
        fig.add_trace(
            go.Scatter(
                x=kle.z,
                y=theoretical_var,
                mode='lines',
                line=dict(dash='dash', color='orange', width=2),
                name='Theoretical'
            ),
            row=3, col=2
        )
        
        fig.update_xaxes(title_text='z', row=3, col=2)
        fig.update_yaxes(title_text='Var[a(z)]', row=3, col=2)
        
        # Layout
        fig.update_layout(
            height=1000,
            showlegend=True,
            title_text=f'<b>KLE Analysis</b> ({cov_type_dropdown.label}, ℓ={length_scale_slider.value:.3f})'
        )
        
        # fig.show()
        import plotly.io as pio
        pio.show(fig, renderer="colab")
        
        # Stats
        print("\n" + "="*70)
        print("STATISTIQUES")
        print("="*70)
        print(f"Covariance type         : {cov_type_dropdown.label}")
        print(f"Length scale            : {length_scale_slider.value:.4f}")
        print(f"Variance                : {variance_slider.value:.2f}")
        print(f"# Points discrétisation : {n_points_slider.value}")
        print(f"# Modes calculés        : {len(kle.eigenvalues)}")
        print(f"\nPremiers eigenvalues :")
        for i in range(min(10, len(kle.eigenvalues))):
            print(f"  λ_{i+1} = {kle.eigenvalues[i]:.6e}")
        
        # Erreur de reconstruction
        error_bound = kle.eigenvalues[rank_slider.value:].sum()
        print(f"\nBorne d'erreur (rank={rank_slider.value}) : {error_bound:.4e}")
# On réinitialise les fonctions liées au clic pour éviter les doublons
compute_button_kle._click_handlers.callbacks = [] 
compute_button_kle.on_click(compute_kle_interactive)

In [6]:
# ============================================================================
# INTERFACE
# ============================================================================

discretization_params = widgets.VBox([
    widgets.HTML("<h3>🔢 Discretization</h3>"),
    n_points_slider,
    quadrature_type_dropdown
])

covariance_params = widgets.VBox([
    widgets.HTML("<h3>📐 Covariance Function</h3>"),
    cov_type_dropdown,
    length_scale_slider,
    variance_slider
])

viz_params = widgets.VBox([
    widgets.HTML("<h3>📊 Visualization</h3>"),
    n_modes_slider,
    n_samples_kle_slider,
    rank_slider,
    compute_button_kle
])

controls_kle = widgets.HBox([discretization_params, covariance_params, viz_params])

display(widgets.VBox([
    widgets.HTML("<h1>🌊 KLE Interactive Explorer</h1>"),
    controls_kle,
    output_kle
]))

compute_kle_interactive()